# AI Code Auditor v3 — Evaluation

Evaluates DeepSeek-Coder-6.7B fine-tuned on Big-Vul v3 (balanced with synthetic data).

### Before running:
1. GPU: T4 x1
2. Attach dataset with: `test.jsonl` + `lora_adapter_v3/`

### What's new in v3:
- Trained on balanced dataset (2,363 samples)
- Added 90 CWE-190 synthetic samples
- Added 50 CWE-416 synthetic samples
- Class balance improved: 3.2x → 1.5x

In [ ]:
!pip install -q transformers==4.40.2 peft==0.10.0 accelerate==0.29.3 bitsandbytes==0.45.3 sacrebleu rouge-score
print('Done')

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['LD_LIBRARY_PATH'] = '/usr/local/cuda/lib64:' + os.environ.get('LD_LIBRARY_PATH', '')
print('Environment set')

In [ ]:
import os, json, re

TOP10 = {'CWE-119','CWE-20','CWE-399','CWE-264','CWE-200',
         'CWE-125','CWE-190','CWE-416','CWE-362','CWE-189'}

# Find paths
TEST_PATH = None
ADAPTER_PATH = None
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        full = os.path.join(root, f)
        if f == 'test.jsonl': TEST_PATH = full
        if f == 'adapter_config.json' and 'checkpoint' not in root:
            ADAPTER_PATH = root

assert TEST_PATH, 'test.jsonl not found'
assert ADAPTER_PATH, 'adapter_config.json not found'
print(f'Test    : {TEST_PATH}')
print(f'Adapter : {ADAPTER_PATH}')

# Load test records — top-10 CWEs only, 100 samples
with open(TEST_PATH) as f:
    all_records = [json.loads(l) for l in f]
test_records = [r for r in all_records if r['cwe'] in TOP10][:100]
print(f'Test samples: {len(test_records)}')

# Show CWE distribution in test set
from collections import Counter
dist = Counter(r['cwe'] for r in test_records)
print('\nTest set CWE distribution:')
for cwe, count in sorted(dist.items(), key=lambda x: -x[1]):
    print(f'  {cwe}: {count}')

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL = 'deepseek-ai/deepseek-coder-6.7b-base'
print(f'CUDA: {torch.cuda.is_available()} | {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb_config,
    device_map={'': 0}, trust_remote_code=True, torch_dtype=torch.float16,
)
base_model.config.use_cache = True
print(f'Base model loaded. VRAM: {torch.cuda.memory_allocated()/1e9:.1f}GB')

In [ ]:
# Load v3 fine-tuned adapter
finetuned_model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
finetuned_model.eval()
print(f'v3 Fine-tuned model ready!')
print(f'Adapter: {ADAPTER_PATH}')

In [ ]:
# Inference helpers
def build_prompt(code):
    return (
        'You are an expert security code auditor.\n'
        'Analyze the following C/C++ code and identify the security vulnerability.\n\n'
        f'```c\n{code}\n```\n\n'
        'Respond with the CWE type first, then explain and provide a secure rewrite.\n'
        'CWE:'
    )

def extract_cwe(text):
    m = re.search(r'CWE-\d+', text)
    return m.group(0) if m else 'Unknown'

def run_inference(model, code, max_new_tokens=200):
    prompt = build_prompt(code)
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=400).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

# Sanity check
test_out = run_inference(finetuned_model, 'void foo(char *s) { char buf[8]; strcpy(buf, s); }')
print(f'Sanity check: {test_out[:120]}')
print(f'Extracted CWE: {extract_cwe(test_out)}')

In [ ]:
# Run v3 fine-tuned inference
from tqdm import tqdm

print('Running v3 FINE-TUNED inference (100 samples)...')
finetuned_results = []
for i, record in enumerate(tqdm(test_records)):
    output = run_inference(finetuned_model, record['vulnerable_code'])
    finetuned_results.append({
        'sample_id': i,
        'ground_truth_cwe': record['cwe'],
        'predicted_cwe': extract_cwe(output),
        'ground_truth_secure': record['secure_code'],
        'predicted_secure': output,
        'raw_output': output,
        'vulnerable_code': record['vulnerable_code'],
    })

with open('/kaggle/working/finetuned_results_v3.jsonl', 'w') as f:
    for r in finetuned_results: f.write(json.dumps(r) + '\n')

ft_acc = sum(1 for r in finetuned_results if r['ground_truth_cwe'] == r['predicted_cwe'])
unknown = sum(1 for r in finetuned_results if r['predicted_cwe'] == 'Unknown')
print(f'\nv3 Fine-tuned CWE Accuracy: {ft_acc}/100 = {ft_acc}%')
print(f'Unknown predictions: {unknown}')

In [ ]:
# Compute all metrics + per-CWE breakdown
import sacrebleu
from rouge_score import rouge_scorer
import numpy as np
from collections import Counter

def compute_metrics(results, name):
    refs  = [r['ground_truth_secure'] for r in results]
    hyps  = [r['predicted_secure']    for r in results]
    gt    = [r['ground_truth_cwe']    for r in results]
    pred  = [r['predicted_cwe']       for r in results]

    bleu   = sacrebleu.corpus_bleu(hyps, [refs]).score
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)
    rougeL = np.mean([scorer.score(r,h)['rougeL'].fmeasure for r,h in zip(refs,hyps)])
    cwe_acc = sum(1 for g,p in zip(gt,pred) if g==p) / len(gt)

    # Per-CWE accuracy
    per_cwe = {}
    for cwe, count in Counter(gt).most_common():
        correct = sum(1 for g,p in zip(gt,pred) if g==cwe and g==p)
        per_cwe[cwe] = {'correct': correct, 'total': count, 'accuracy': round(correct/count, 3)}

    print(f'\n{"="*55}')
    print(f'  {name}')
    print(f'{"="*55}')
    print(f'  BLEU-4        : {bleu:.2f}')
    print(f'  ROUGE-L       : {rougeL:.3f}')
    print(f'  CWE Accuracy  : {cwe_acc:.1%} ({sum(1 for g,p in zip(gt,pred) if g==p)}/{len(gt)})')
    print(f'  Unknown preds : {sum(1 for p in pred if p=="Unknown")}')
    print(f'\n  Per-CWE accuracy:')
    for cwe, stats in sorted(per_cwe.items(), key=lambda x: -x[1]['total']):
        bar = '█' * stats['correct'] + '░' * (stats['total'] - stats['correct'])
        print(f'    {cwe}: {stats["correct"]}/{stats["total"]} ({stats["accuracy"]:.0%}) {bar}')

    return {
        'model': name,
        'bleu4': round(bleu, 2),
        'rougeL': round(rougeL, 3),
        'cwe_accuracy': round(cwe_acc, 3),
        'unknown_count': sum(1 for p in pred if p=='Unknown'),
        'correct': sum(1 for g,p in zip(gt,pred) if g==p),
        'total': len(gt),
        'per_cwe': per_cwe
    }

ft_metrics = compute_metrics(finetuned_results, 'v3 Fine-tuned (QLoRA DeepSeek-6.7B + Synthetic)')

# Save metrics
with open('/kaggle/working/evaluation_metrics_v3.json', 'w') as f:
    json.dump({'finetuned_v3': ft_metrics}, f, indent=2)
print('\nSaved evaluation_metrics_v3.json')

In [ ]:
# v2 vs v3 Comparison
print('\n' + '='*55)
print('  v2 vs v3 COMPARISON')
print('='*55)

# v2 results (hardcoded from your saved results)
v2_metrics = {
    'cwe_accuracy': 0.26,
    'bleu4': 5.69,
    'rougeL': 0.270,
    'unknown_count': 31,
    'per_cwe': {
        'CWE-119': {'correct': 2, 'total': 27},
        'CWE-20':  {'correct': 7, 'total': 25},
        'CWE-399': {'correct': 6, 'total': 14},
        'CWE-125': {'correct': 6, 'total': 10},
        'CWE-200': {'correct': 0, 'total': 8},
        'CWE-264': {'correct': 3, 'total': 5},
        'CWE-190': {'correct': 0, 'total': 4},
        'CWE-416': {'correct': 0, 'total': 4},
        'CWE-189': {'correct': 0, 'total': 2},
        'CWE-362': {'correct': 0, 'total': 1}
    }
}

v3_acc = ft_metrics['cwe_accuracy']
v2_acc = v2_metrics['cwe_accuracy']
improvement = ((v3_acc - v2_acc) / v2_acc) * 100

print(f'\n  Overall CWE Accuracy:')
print(f'    v2: {v2_acc:.1%} ({int(v2_acc*100)}/100)')
print(f'    v3: {v3_acc:.1%} ({ft_metrics["correct"]}/100)')
print(f'    Change: {improvement:+.1f}%  {"✅ IMPROVED" if improvement > 0 else "❌ REGRESSED"}')

print(f'\n  BLEU-4:')
print(f'    v2: {v2_metrics["bleu4"]:.2f}')
print(f'    v3: {ft_metrics["bleu4"]:.2f}')

print(f'\n  ROUGE-L:')
print(f'    v2: {v2_metrics["rougeL"]:.3f}')
print(f'    v3: {ft_metrics["rougeL"]:.3f}')

print(f'\n  Unknown predictions:')
print(f'    v2: {v2_metrics["unknown_count"]}/100')
print(f'    v3: {ft_metrics["unknown_count"]}/100')

print(f'\n  Per-CWE Comparison (v2 → v3):')
target_cwes = ['CWE-190', 'CWE-416', 'CWE-189', 'CWE-362', 'CWE-125']
for cwe in target_cwes:
    v2 = v2_metrics['per_cwe'].get(cwe, {'correct': 0, 'total': 0})
    v3 = ft_metrics['per_cwe'].get(cwe, {'correct': 0, 'total': 0, 'accuracy': 0})
    v2_a = v2['correct']/v2['total'] if v2['total'] > 0 else 0
    v3_a = v3.get('accuracy', 0)
    diff = v3_a - v2_a
    status = '✅' if diff > 0 else ('➡️' if diff == 0 else '❌')
    print(f'    {cwe}: {v2_a:.0%} → {v3_a:.0%} ({diff:+.0%}) {status}')

print('='*55)

In [ ]:
# Generate comparison chart
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot 1: Overall metrics comparison
metrics_names = ['CWE Accuracy', 'BLEU-4 / 10', 'ROUGE-L']
v2_vals = [v2_metrics['cwe_accuracy'], v2_metrics['bleu4']/10, v2_metrics['rougeL']]
v3_vals = [ft_metrics['cwe_accuracy'], ft_metrics['bleu4']/10, ft_metrics['rougeL']]

x = np.arange(len(metrics_names))
w = 0.35
axes[0].bar(x - w/2, v2_vals, w, label='v2 (Big-Vul only)', color='steelblue', alpha=0.8)
axes[0].bar(x + w/2, v3_vals, w, label='v3 (+ Synthetic)', color='coral', alpha=0.8)
axes[0].set_title('Overall Metrics: v2 vs v3', fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics_names)
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)
axes[0].set_ylim(0, 0.6)

# Plot 2: Per-CWE accuracy for target classes
target_cwes = ['CWE-190', 'CWE-416', 'CWE-189', 'CWE-362', 'CWE-125']
v2_cwe_vals = []
v3_cwe_vals = []
for cwe in target_cwes:
    v2c = v2_metrics['per_cwe'].get(cwe, {'correct': 0, 'total': 1})
    v3c = ft_metrics['per_cwe'].get(cwe, {'correct': 0, 'total': 1, 'accuracy': 0})
    v2_cwe_vals.append(v2c['correct']/v2c['total'] if v2c['total'] > 0 else 0)
    v3_cwe_vals.append(v3c.get('accuracy', 0))

x = np.arange(len(target_cwes))
axes[1].bar(x - w/2, v2_cwe_vals, w, label='v2', color='steelblue', alpha=0.8)
axes[1].bar(x + w/2, v3_cwe_vals, w, label='v3', color='coral', alpha=0.8)
axes[1].set_title('Target CWE Accuracy: v2 vs v3', fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(target_cwes, rotation=15)
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)
axes[1].set_ylim(0, 1.0)

plt.suptitle('AI Code Auditor: v2 vs v3 Performance', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/v2_v3_comparison.png', dpi=150, bbox_inches='tight')
plt.close()
print('✅ Saved v2_v3_comparison.png')

In [ ]:
# Download all results
from IPython.display import FileLink, display
print('📥 Download your results:')
display(FileLink('evaluation_metrics_v3.json'))
display(FileLink('finetuned_results_v3.jsonl'))
display(FileLink('v2_v3_comparison.png'))